# Lab 02: Waiting Time at a Bus Stop

*SF250 Probability Theory and Statistics. Written in C, run from this notebook.*


# Waiting Time at a Bus Stop

Buses arrive on average every T minutes. How long does a passenger who shows
up at a random moment wait? This program compares two kinds of bus service
with the same average rate and shows that the average wait is not the same.

1. **Regular buses**: one bus exactly every T minutes. The wait is
   Uniform[0, T], so the mean wait is T/2.
2. **Random buses**: arrivals follow a Poisson process with rate 1/T. The wait
   is Exponential with mean T.

Same rate, but the random service makes you wait twice as long on average.

## Build and run

```
make
./waiting_times
```

The program asks two questions:

| Prompt | Meaning |
|---|---|
| `Average bus interval T` | average minutes between buses, for example 15 |
| `Number of samples` | how many simulated passengers, for example 100000 |

The random seed is fixed (250), so the same inputs always give the same output.

## How it works

For each simulated passenger two waiting times are drawn from one uniform
random number `u` in (0, 1):

- regular: `T * u`, which is Uniform[0, T]
- random: `-T * ln(u)`, which is Exponential with mean T (a uniform number
  pushed through `-ln` gives an exponential one)

Each wait is dropped into a histogram bin of width T/10. There are 30 bins
covering 0 to 3T, and the last bin also collects any wait of 3T or more. The
program prints the two histograms with the sample mean next to the theory.

## Makefile shortcuts

```
make                        build ./waiting_times
make run                    build and run, answering 15 and 100000 for you
make run INPUT='10\n5000\n'  answer with your own two numbers
make clean                  remove the program
```

## Example

```
$ printf '15\n100000\n' | ./waiting_times

1) Regular buses: Uniform[0,T]  (mean = 7.517)
  0.00-  1.50 |################################################ 9886
  1.50-  3.00 |################################################# 10036
  ...
   theory T/2 = 7.500

2) Random buses: Exponential(1/T)  (mean = 14.998)
  ...
   theory T = 15.000   (last bar includes waits >= 3T)
```

How to read it:
- The first histogram is flat, so every wait between 0 and T is equally likely.
- The second falls off like `e^(-x/T)`: short waits are common, but a long
  tail of 30, 40 minutes or more pulls the mean up to T.

Try `T = 15` with 100 samples and then 1,000,000 samples: the means move
closer to 7.5 and 15.


## Slide references

ชื่อย่อสไลด์ (เลขหลังชื่อคือเลขหน้า "Slide" มุมขวาล่าง):

| ย่อ | ชุดสไลด์ |
|---|---|
| **PF** | Probability Fundamentals |
| **DD** | Discrete Probability Distributions |
| **CD** | Continuous Distributions |
| **MGF** | Moment Generating Functions |
| **CM** | Combining and Manipulating Distributions |
| **TS** | Testing and Statistical Significance |

## 3. `waiting_bus`: เวลารอรถประจำทาง

**คืออะไร:** รถมาเฉลี่ยทุก T นาที เทียบเวลารอเฉลี่ยของคนที่มาถึงป้ายแบบสุ่ม ในสองกรณี
(1) รถมาตรงเวลาสม่ำเสมอ (2) รถมาแบบสุ่ม (Poisson)

**ทำงานยังไง:** สุ่มเวลารอ n ครั้งต่อกรณี กรณี 1 ใช้ `T·u` (Uniform[0,T]) กรณี 2 ใช้ `−T·ln(u)` (Exponential) แล้ววาดฮิสโตแกรมและเทียบค่าเฉลี่ยกับทฤษฎี T/2 กับ T
คำตอบที่น่าแปลกคือ อัตรามาเท่ากันแต่เวลารอเฉลี่ยต่างกัน 2 เท่า

| ส่วนของโปรแกรม | อ้างอิงสไลด์ |
|---|---|
| โจทย์นี้ตรงๆ (รถทุก 15 นาที, Uniform ได้ 7.5 กับ Exponential ได้ 15) | CM 2 |
| รถมาแบบสุ่ม = Poisson | DD 13-15 |
| เวลาระหว่างเหตุการณ์ Poisson = Exponential, ที่มาของ λe^(−λt) | CD 6-7 |
| mean = 1/λ, var = 1/λ² | CD 8 |
| `−T·ln(u)` สร้าง Exponential จาก Uniform (y = −log x ได้ Ex(1)) | CM 15-16 |
| คูณด้วย T เพื่อเปลี่ยนหน่วยเวลา (Jacobian) | CM 12 |

ทุกส่วนอยู่ในสไลด์


## Running C in a notebook

This notebook uses the normal Python kernel. A cell that starts with `%%writefile` saves a C source file, and a cell that starts with `!` runs a shell command (`make`, then the compiled program). It needs `gcc` and `make`, which Google Colab already has. On a local Jupyter, install them first (for example `sudo apt install build-essential`). The C code is the same code as in the repository, written in 42 Norm style.


In [1]:
!mkdir -p waiting_bus

In [2]:
!gcc --version | head -1

gcc (GCC) 16.2.1 20261029 (Red Hat 16.2.1-3)


## Source code

One cell per file. Run them all to recreate the project.

In [3]:
%%writefile waiting_bus/waiting_times.c
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define BINS 30

static double	u(void)
{
	return ((rand() + 1.0) / (RAND_MAX + 2.0));
}

static void	hist(const char *name, const int *h, double w, double mean)
{
	int	max;
	int	i;
	int	j;

	max = 1;
	i = -1;
	while (++i < BINS)
		if (h[i] > max)
			max = h[i];
	printf("\n%s  (mean = %.3f)\n", name, mean);
	i = -1;
	while (++i < BINS)
	{
		printf("%6.2f-%6.2f |", i * w, (i + 1) * w);
		j = -1;
		while (++j < h[i] * 50 / max)
			putchar('#');
		printf(" %d\n", h[i]);
	}
}

static void	add(int *h, double x, double w)
{
	int	i;

	i = (int)(x / w);
	if (i >= BINS)
		i = BINS - 1;
	h[i]++;
}

static void	sim(double period, int n, int h[2][BINS], double sum[2])
{
	int		i;
	double	a;
	double	b;

	i = -1;
	sum[0] = 0;
	sum[1] = 0;
	srand(250);
	while (++i < n)
	{
		a = period * u();
		b = -period * log(u());
		sum[0] += a;
		sum[1] += b;
		add(h[0], a, period / 10);
		add(h[1], b, period / 10);
	}
}

int	main(void)
{
	double	period;
	int		n;
	int		h[2][BINS];
	double	sum[2];

	printf("Average bus interval T (minutes, e.g. 15): ");
	if (scanf("%lf", &period) != 1 || period <= 0)
		return (1);
	printf("Number of samples (e.g. 100000): ");
	if (scanf("%d", &n) != 1 || n <= 0)
		return (1);
	memset(h, 0, sizeof(h));
	sim(period, n, h, sum);
	hist("1) Regular buses: Uniform[0,T]", h[0], period / 10, sum[0] / n);
	printf("   theory T/2 = %.3f\n", period / 2);
	hist("2) Random buses: Exponential(1/T)", h[1], period / 10, sum[1] / n);
	printf("   theory T = %.3f   (last bar includes waits >= 3T)\n", period);
	return (0);
}


Writing waiting_bus/waiting_times.c


In [4]:
%%writefile waiting_bus/Makefile
NAME = waiting_times
SRCS = waiting_times.c
CFLAGS = -Wall -Wextra -O2
INPUT = 15\n100000\n

all: $(NAME)

$(NAME): $(SRCS)
	$(CC) $(CFLAGS) $(SRCS) -o $(NAME) -lm

# The program asks two questions (average interval T, number of samples).
# make run                         answers 15 and 100000
# make run INPUT='10\n5000\n'      answers with your own numbers
run: $(NAME)
	printf '$(INPUT)' | ./$(NAME)

clean:
	rm -f $(NAME)

re: clean all

.PHONY: all run clean re


Writing waiting_bus/Makefile


## Build

`make` compiles every `.c` file into one program.

In [5]:
!cd waiting_bus && make

cc -Wall -Wextra -O2 waiting_times.c -o waiting_times -lm


## Run it

Buses every 15 minutes on average, 100000 passengers. The program asks two questions, so we pipe the answers in:

In [6]:
!cd waiting_bus && printf '15\n100000\n' | ./waiting_times

Average bus interval T (minutes, e.g. 15): Number of samples (e.g. 100000): 
1) Regular buses: Uniform[0,T]  (mean = 7.517)
  0.00-  1.50 |################################################ 9886
  1.50-  3.00 |################################################# 10036
  3.00-  4.50 |################################################ 9908
  4.50-  6.00 |################################################# 10043
  6.00-  7.50 |################################################ 9988
  7.50-  9.00 |################################################ 9925
  9.00- 10.50 |################################################ 10016
 10.50- 12.00 |################################################ 9973
 12.00- 13.50 |################################################## 10232
 13.50- 15.00 |################################################ 9993
 15.00- 16.50 | 0
 16.50- 18.00 | 0
 18.00- 19.50 | 0
 19.50- 21.00 | 0
 21.00- 22.50 | 0
 22.50- 24.00 | 0
 24.00- 25.50 | 0
 25.50- 27.00 | 0
 27.00- 28.50 | 0
 28.50- 30.00 | 

A different interval and a small sample (means are noisier):

In [7]:
!cd waiting_bus && printf '10\n500\n' | ./waiting_times

Average bus interval T (minutes, e.g. 15): Number of samples (e.g. 100000): 
1) Regular buses: Uniform[0,T]  (mean = 5.113)
  0.00-  1.00 |#################################### 42
  1.00-  2.00 |##################################### 44
  2.00-  3.00 |############################################ 52
  3.00-  4.00 |########################################### 50
  4.00-  5.00 |########################################### 51
  5.00-  6.00 |############################################## 54
  6.00-  7.00 |############################################ 52
  7.00-  8.00 |################################################## 58
  8.00-  9.00 |##################################### 44
  9.00- 10.00 |############################################# 53
 10.00- 11.00 | 0
 11.00- 12.00 | 0
 12.00- 13.00 | 0
 13.00- 14.00 | 0
 14.00- 15.00 | 0
 15.00- 16.00 | 0
 16.00- 17.00 | 0
 17.00- 18.00 | 0
 18.00- 19.00 | 0
 19.00- 20.00 | 0
 20.00- 21.00 | 0
 21.00- 22.00 | 0
 22.00- 23.00 | 0
 23.00- 24.00 | 0
 24.00- 2

## Try it yourself

Change the numbers in a command above and re-run the cell. The arguments are described in the usage section at the top of this notebook.